# 02 — Análise por área de conhecimento

Este notebook contém  as análises apresentadas no TCC para comparar **TF-IDF** e **Qwen3.5-4B** em `nDCG@10`:

1. diferença média de `nDCG@10` por área;
2. diferença de `nDCG@10` por autor;
3. comparação autor a autor por área (TF-IDF vence / empate / Qwen3.5-4B vence).

A análise utiliza o conjunto completo de autores e as 10 áreas com maior número de pesquisadores. O critério de empate na comparação autor a autor é `|Δ nDCG@10| < 0.01`.

## 1. Configuração dos caminhos

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data").exists() or (p / "src").exists()), current)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ARQ_AREA_PREDOMINANTE = DATA_DIR / "area_predominante_por_autor.csv"
ARQ_TFIDF_METRICAS = RESULTS_DIR / "baselines" / "tfidf" / "metricas_tfidf_por_autor.csv"
ARQ_QWEN35_4_METRICAS = RESULTS_DIR / "qwen_few_shot" / "qwen3_5_4b" / "metricas_qwen3_5_4B_por_autor.csv"

for nome, caminho in {
    "Área predominante": ARQ_AREA_PREDOMINANTE,
    "TF-IDF": ARQ_TFIDF_METRICAS,
    "Qwen3.5-4B": ARQ_QWEN35_4_METRICAS,
}.items():
    print(f"{nome:20s} -> {'OK' if caminho.exists() else 'não encontrado'}")

## 2. Carregamento e preparação dos dados

In [ ]:
METRICA = "nDCG@10"
LIMIAR_EMPATE = 0.01


def carregar_csv(caminho):
    return pd.read_csv(caminho, sep=None, engine="python")


def normalizar_id(x):
    x = str(x).strip()
    return x if x.startswith("ID_") else f"ID_{x}"


df_area = carregar_csv(ARQ_AREA_PREDOMINANTE)
df_area.rename(columns={"\ufeffAutor": "Autor"}, inplace=True)
df_area["Autor"] = df_area["Autor"].apply(normalizar_id)
df_area = df_area[df_area["Area_Predominante"].notna()].copy()

# Mantém também autores cuja área predominante foi definida por desempate,
# como no código utilizado na análise final.
top10_areas = (
    df_area.groupby("Area_Predominante")["Autor"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

df_tfidf = carregar_csv(ARQ_TFIDF_METRICAS)
df_qwen = carregar_csv(ARQ_QWEN35_4_METRICAS)

for df in (df_tfidf, df_qwen):
    df["Autor"] = df["Autor"].apply(normalizar_id)
    if METRICA not in df.columns:
        raise ValueError(f"Métrica {METRICA!r} não encontrada. Colunas: {list(df.columns)}")

# Junta as métricas dos dois sistemas autor a autor.
df_delta_autor = df_tfidf[["Autor", METRICA]].merge(
    df_qwen[["Autor", METRICA]],
    on="Autor",
    suffixes=("_TFIDF", "_Qwen"),
)

df_delta_autor["Delta_nDCG@10"] = (
    df_delta_autor["nDCG@10_Qwen"] - df_delta_autor["nDCG@10_TFIDF"]
)

print(f"Autores comparados: {len(df_delta_autor)}")
print("Top 10 áreas:")
display(top10_areas.rename("Numero_Autores").reset_index())

## 3. Diferença média de nDCG@10 por área

In [ ]:
df_por_area = df_delta_autor.merge(
    df_area[["Autor", "Area_Predominante"]],
    on="Autor",
    how="inner",
)
df_por_area = df_por_area[
    df_por_area["Area_Predominante"].isin(top10_areas.index)
].copy()

df_resumo_areas = (
    df_por_area.groupby("Area_Predominante")[["nDCG@10_TFIDF", "nDCG@10_Qwen"]]
    .mean()
    .reset_index()
    .rename(columns={
        "Area_Predominante": "Area",
        "nDCG@10_TFIDF": "TF-IDF",
        "nDCG@10_Qwen": "Qwen3.5-4B",
    })
)

df_resumo_areas["Autores"] = df_resumo_areas["Area"].map(top10_areas)
df_resumo_areas["Delta"] = df_resumo_areas["Qwen3.5-4B"] - df_resumo_areas["TF-IDF"]
df_resumo_areas["Melhor"] = df_resumo_areas["Delta"].apply(
    lambda x: "Qwen3.5-4B" if x > 0 else "TF-IDF"
)

display(df_resumo_areas.sort_values("Autores", ascending=False).round(4))

# Figura 1
plt.rcdefaults()
df_delta_area = df_resumo_areas.sort_values("Delta", ascending=True).copy()
fig, ax = plt.subplots(figsize=(10, 6))
cores = ["tab:orange" if x > 0 else "tab:blue" for x in df_delta_area["Delta"]]
ax.barh(df_delta_area["Area"], df_delta_area["Delta"], color=cores)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Delta nDCG@10 (Qwen3.5-4B − TF-IDF)")
ax.set_ylabel("Área do conhecimento")
ax.set_title("Diferença de nDCG@10 por área")
ax.legend(handles=[
    Patch(color="tab:blue", label="TF-IDF"),
    Patch(color="tab:orange", label="Qwen3.5-4B"),
], title="Melhor desempenho")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "delta_ndcg10_por_area.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Diferença de nDCG@10 por autor

In [ ]:
df_delta_autor_plot = df_delta_autor.sort_values("Delta_nDCG@10").reset_index(drop=True)

plt.rcdefaults()
fig, ax = plt.subplots(figsize=(18, 7))
cores = [
    "tab:blue" if x < 0 else "tab:orange"
    for x in df_delta_autor_plot["Delta_nDCG@10"]
]

ax.bar(
    range(len(df_delta_autor_plot)),
    df_delta_autor_plot["Delta_nDCG@10"],
    color=cores,
    width=0.8,
)
ax.axhline(0, color="tab:blue", linewidth=1)
ax.set_title("Diferença de nDCG@10 por autor — Qwen3.5-4B vs TF-IDF")
ax.set_xlabel("Autores")
ax.set_ylabel("Delta nDCG@10 (Qwen3.5-4B - TF-IDF)")
ax.set_xticks([])
ax.grid(False)
ax.legend(handles=[
    Patch(color="tab:blue", label="TF-IDF"),
    Patch(color="tab:orange", label="Qwen3.5-4B"),
])
plt.tight_layout()
fig.savefig(FIGURES_DIR / "delta_ndcg10_por_autor.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Comparação autor a autor por área

In [ ]:
df_winloss = df_delta_autor.merge(
    df_area[["Autor", "Area_Predominante"]],
    on="Autor",
    how="inner",
)


def classificar(delta):
    if abs(delta) < LIMIAR_EMPATE:
        return "Empate"
    if delta > 0:
        return "Qwen3.5-4B"
    return "TF-IDF"


df_winloss["Resultado"] = df_winloss["Delta_nDCG@10"].apply(classificar)

top10 = (
    df_winloss.groupby("Area_Predominante")["Autor"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
)

df_winloss_top10 = df_winloss[df_winloss["Area_Predominante"].isin(top10)].copy()
df_tabela_winloss = pd.crosstab(
    df_winloss_top10["Area_Predominante"],
    df_winloss_top10["Resultado"],
)

for col in ["TF-IDF", "Empate", "Qwen3.5-4B"]:
    if col not in df_tabela_winloss.columns:
        df_tabela_winloss[col] = 0

df_tabela_winloss = df_tabela_winloss[["TF-IDF", "Empate", "Qwen3.5-4B"]].reset_index()
df_tabela_winloss["Autores"] = df_tabela_winloss[["TF-IDF", "Empate", "Qwen3.5-4B"]].sum(axis=1)
df_tabela_winloss = df_tabela_winloss.rename(columns={
    "Area_Predominante": "Area",
    "TF-IDF": "TF-IDF_Vence",
    "Empate": "Empates",
    "Qwen3.5-4B": "Qwen3.5-4B_Vence",
})

df_tabela_winloss["TF-IDF_%"] = 100 * df_tabela_winloss["TF-IDF_Vence"] / df_tabela_winloss["Autores"]
df_tabela_winloss["Empate_%"] = 100 * df_tabela_winloss["Empates"] / df_tabela_winloss["Autores"]
df_tabela_winloss["Qwen3.5-4B_%"] = 100 * df_tabela_winloss["Qwen3.5-4B_Vence"] / df_tabela_winloss["Autores"]

df_tabela_winloss = df_tabela_winloss.sort_values("Autores", ascending=False).reset_index(drop=True)
display(df_tabela_winloss.round(2))

# Figura 3
df_plot = df_tabela_winloss.sort_values("Autores", ascending=True).copy()
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(df_plot["Area"], df_plot["TF-IDF_%"], label="TF-IDF vence")
ax.barh(
    df_plot["Area"],
    df_plot["Empate_%"],
    left=df_plot["TF-IDF_%"],
    label="Empate",
)
ax.barh(
    df_plot["Area"],
    df_plot["Qwen3.5-4B_%"],
    left=df_plot["TF-IDF_%"] + df_plot["Empate_%"],
    label="Qwen3.5-4B vence",
)
ax.set_xlabel("Autores (%)")
ax.set_ylabel("Área do conhecimento")
ax.set_title("Comparação autor a autor de nDCG@10 por área")
ax.set_xlim(0, 100)
ax.legend(title="Resultado")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "winloss_ndcg10_por_area.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Saídas

O notebook salva as três figuras utilizadas nesta análise:

- `figures/delta_ndcg10_por_area.png`
- `figures/delta_ndcg10_por_autor.png`
- `figures/winloss_ndcg10_por_area.png`
